# SSN Builder — sequence-based (mmseqs)

Builds **Sequence Similarity Networks (SSN)** from a FASTA file using mmseqs2.  
For structure-based networks (foldseek / PDB), see `build_stsn.ipynb`.

Run cells top to bottom, skipping optional ones as needed.

## Setup

In [1]:
import sys, logging
from pathlib import Path

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("ssn")

In [2]:
# Shared paths — edit these before running anything else
INPUT_FASTA = Path("/Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/data/GH43.fa")
PREFIX      = "GH43"
OUTPUT_DIR  = Path("/Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/GH43_0.4-0.4/")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---
## Step 1 — Redundancy reduction *(optional)*

Run **one** of the two cells below.

### 1a · Cluster with mmseqs

In [ ]:
from mmseqs import easy_cluster

result = easy_cluster(
    fasta      = INPUT_FASTA,
    output_dir = OUTPUT_DIR / "cluster",
    prefix     = PREFIX,
    min_seq_id = 0.8,
    coverage   = 0.8,
    cov_mode   = 0,    # 0 = bidirectional, 1 = query, 2 = target
    log        = log,
)
rep_fasta   = result["rep_seq_fasta"]
cluster_tsv = result["cluster_tsv"]
print(f"rep_fasta:   {rep_fasta}")
print(f"cluster_tsv: {cluster_tsv}")

### 1b · Skip clustering — use full input FASTA

In [ ]:
rep_fasta   = INPUT_FASTA
cluster_tsv = None
print(f"Using full input FASTA: {rep_fasta}")

---
## Step 2 — All-vs-all sequence similarity search (mmseqs)

Run **one** of the two cells below.

### 2a · Run mmseqs easy-search

In [ ]:
from mmseqs import easy_search

result = easy_search(
    fasta       = rep_fasta,
    output_dir  = OUTPUT_DIR / "search",
    prefix      = PREFIX,
    evalue      = 1e-5,
    sensitivity = 7.5,
    min_seq_id  = 0.4,
    coverage    = 0.4,
    cov_mode    = 1,
    log         = log,
)
m8_path = result["m8"]
print(f"m8: {m8_path}")

### 2b · Use existing .m8 file

In [3]:
m8_path = Path("/Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/GH43_0.4-0.4/search/GH43.m8")

---
## Load similarity data

In [4]:
from plot_ssn import load_data

df = load_data(m8_path)
df.head()

15:19:39  INFO      NumExpr defaulting to 8 threads.
15:19:40  INFO      START load similarity table /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/GH43_0.4-0.4/search/GH43.m8
15:19:40  INFO      END   load similarity table /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/GH43_0.4-0.4/search/GH43.m8: 0.49s
15:19:40  INFO      Loaded 1533648 rows from /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/GH43_0.4-0.4/search/GH43.m8
15:19:40  INFO      START remove self-hits
15:19:40  INFO      END   remove self-hits: 0.10s
15:19:40  INFO      Removed self-hits where qseqid == sseqid: 11390 rows
15:19:40  INFO      After removing self-hits: 1522258 rows


Loaded data: 1522258 edges


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
1,ABK48312.1,QYK03049.1,78.0,341,75,0,15,355,14,353,9.886000e-187,578
2,ABK48312.1,ACB74675.1,67.9,336,106,0,14,349,9,340,2.723000e-158,496
3,ABK48312.1,XXN22993.1,66.3,333,111,0,16,348,8,338,9.591000e-158,494
4,ABK48312.1,XAW86689.1,66.5,341,110,0,21,349,17,357,3.057000e-156,490
5,ABK48312.1,BET65593.1,62.8,342,126,0,12,352,3,344,1.497000e-152,479


---
## Load annotations *(optional)*

Skip this cell to produce uncolored networks.

In [8]:
import pandas as pd
from plot_ssn import build_color_lookup, _read_meta

META_FILE  = Path("/Users/maartenboneschansker/Documents/PhD/projects/cazyme_data_collection/2026-09-24_download/consolidated/characterized_ALL.tsv")
ID_COL     = "GenBank"          # column matching sequence IDs in the m8 file
COLOR_COLS = ["EC"]     # one plot is produced per column

meta = _read_meta(META_FILE, ID_COL)
print(f"Metadata: {len(meta)} rows  |  columns: {list(meta.columns)}")

color_lookups = {}
for col in COLOR_COLS:
    color_lookups[col] = build_color_lookup(meta, ID_COL, col, clusters=None)
    print(f"  '{col}': {len(color_lookups[col])} annotated entries")

Metadata: 12279 rows  |  columns: ['cazy_family', 'Domain', 'Protein Name', 'EC', 'Reference', 'Organism', 'GenBank', 'Uniprot', 'PDB/3D', 'Subfamily']
  'EC': 12170 annotated entries


15:25:23  INFO      TemporaryDirectory.cleanup() worked.
--- Logging error ---
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "/opt/miniconda3/lib/python3.12/site-packages/ipykernel/iostream.py", line 760, in write
    self._schedule_flush()
  File "/opt/miniconda3/lib/python3.12/site-packages/ipykernel/iostream.py", line 656, in _schedule_flush
    self.pub_thread.schedule(_schedule_in_thread)
  File "/opt/miniconda3/lib/python3.12/site-packages/ipykernel/iostream.py", line 339, in schedule
    self._event_pipe.send(b"")
  File "/opt/miniconda3/lib/python3.12/site-packages/zmq/sugar/socket.py", line 698, in send
    return super().send(data, flags=flags, copy=copy, track=track)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "zmq/backend/cython/_zmq.py", line 1152, in zmq.backend.cython._zmq.Socket.send
  File "zmq/backend/cython/_zmq.py", line 1

---
## Step 3a — Sequence Similarity Network (SSN)

In [ ]:
# IDs to highlight with a red circle in the SSN plot.
# Add/remove IDs as needed; set to [] to disable highlighting.
HIGHLIGHT_IDS = [
    "NCC5025_ctg.s1.000000F_882",
    "R11_18m_3_chromosome_248",
]

In [ ]:
import json
import kaleido
from plot_ssn import create_graph, exclude_singleton_components, get_layout, plot_ssn, graph_stats

THRESHOLDS         = [0.6]       # pident thresholds (0–1)
NODE_SIZE          = 4
LAYOUT             = "component_grid" # "component_grid" | "fr" | "lgl" | "kk"
EXCLUDE_SINGLETONS = True

# Use color_lookups from the annotations cell, or {} for uncolored plots
_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]
_highlight  = locals().get("HIGHLIGHT_IDS", [])

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
ssn_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = create_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = exclude_singleton_components(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} edges")
    layout_coords = get_layout(g, LAYOUT)
    ssn_stats.append(graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_ssn_t{thresh}_{col_lbl}.html"
        plot_ssn(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col,
                 highlight_ids=_highlight)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_ssn_stats.json"
stats_path.write_text(json.dumps(ssn_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(ssn_stats)

15:26:23  INFO      START deduplicate undirected edges
15:26:23  INFO      Deduplicated reciprocal/parallel edges: 202743 -> 111524
15:26:23  INFO      END   deduplicate undirected edges: 0.09s
15:26:23  INFO      START create edge list
15:26:23  INFO      END   create edge list: 0.02s
15:26:23  INFO      START build igraph
15:26:23  INFO      TemporaryDirectory.cleanup() worked.
15:26:23  INFO      shutil.rmtree worked.
15:26:23  INFO      END   build igraph: 0.05s
15:26:23  INFO      START attach edge pident weights
15:26:23  INFO      END   attach edge pident weights: 0.00s
15:26:23  INFO      START compute kk layout (10166 nodes, 111524 edges)
15:27:40  INFO      TemporaryDirectory.cleanup() worked.


t=0.6: 10166 nodes, 111524 edges


15:27:40  INFO      shutil.rmtree worked.
15:27:40  INFO      END   compute kk layout (10166 nodes, 111524 edges): 76.99s
15:27:40  INFO      START prepare node colors for EC
15:27:40  INFO      END   prepare node colors for EC: 0.00s
15:27:40  INFO      START build edge trace for EC
15:27:41  INFO      END   build edge trace for EC: 0.72s
15:27:41  INFO      START build node traces for EC (discrete)
15:27:46  INFO      END   build node traces for EC (discrete): 4.54s
15:27:46  INFO      START assemble figure for EC
15:27:46  INFO      END   assemble figure for EC: 0.80s
15:27:46  INFO      Skipping HTML for EC: 10166 vertices > 1000
15:27:46  INFO      START write PNG /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/GH43_0.4-0.4/plots/GH43_ssn_t0.6_EC.png


  Skipping HTML (color=EC): 10166 vertices > 1000


15:27:47  INFO      Chromium init'ed with kwargs {}
15:27:47  INFO      Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
15:27:47  INFO      Temp directory created: /var/folders/y9/03cy15vn41jfykwhlpv0fx300000gn/T/tmp3mwd3oeg.
15:27:47  INFO      Opening browser.
15:27:47  INFO      Temp directory created: /var/folders/y9/03cy15vn41jfykwhlpv0fx300000gn/T/tmp8momvca8.
15:27:47  INFO      Temporary directory at: /var/folders/y9/03cy15vn41jfykwhlpv0fx300000gn/T/tmp8momvca8
15:27:47  INFO      Conforming 1 to file:///var/folders/y9/03cy15vn41jfykwhlpv0fx300000gn/T/tmp3mwd3oeg/index.html
15:27:49  INFO      Getting tab from queue (has 1)
15:27:49  INFO      Got CEAF
15:27:53  INFO      Reloading tab CEAF before return.
15:27:53  INFO      Putting tab CEAF back (queue size: 0).
15:27:53  INFO      Waiting for all cleanups to finish.
15:27:53  INFO      Exiting Kaleido.
15:27:53  INFO      TemporaryDirectory.cleanup() worked.
15:27:53  INFO      shutil.rmtree 

  Saved: /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/GH43_0.4-0.4/plots/GH43_ssn_t0.6_EC.png

Stats → /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/GH43_0.4-0.4/plots/GH43_ssn_stats.json


,threshold,nodes,edges,n_components,largest_component,singletons,annotated_nodes,annotated_percent
0,0.6,10166,111524,622,3061,0,92,0.9


---
## Step 3b — Minimum Spanning Tree / StSN *(optional)*

Reduces each connected component to its spanning tree (highest-similarity edges only).  
Useful for structure-based networks (foldseek) or for cleaner visualizations of dense SSNs.

In [ ]:
from plot_mst import (
    build_mst_graph,
    exclude_singleton_components as mst_exclude_singletons,
    get_layout as mst_get_layout,
    plot_mst,
    graph_stats as mst_graph_stats,
)

THRESHOLDS         = [0.4, 0.6]       # pident thresholds (0–1)
NODE_SIZE          = 2
LAYOUT             = "component_grid"
EXCLUDE_SINGLETONS = True

_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
mst_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = build_mst_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = mst_exclude_singletons(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} MST edges")
    layout_coords = mst_get_layout(g, LAYOUT)
    mst_stats.append(mst_graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_t{thresh}_{col_lbl}.html"
        plot_mst(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_stats.json"
stats_path.write_text(json.dumps(mst_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(mst_stats)

---
## Results

In [ ]:
from IPython.display import display, HTML

html_files = sorted((OUTPUT_DIR / "plots").glob("*.html"))
items = "".join(f'<li><a href="{f.resolve()}" target="_blank">{f.name}</a></li>' for f in html_files)
display(HTML(f"<ul>{items}</ul>") if items else HTML("<i>No plots found — run cells above first.</i>"))